#### Chroma
Chroma is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

https://python.langchain.com/v0.2/docs/integrations/vectorstores/

In [2]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness 

In [4]:
## split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(data)

In [5]:
embedding = OllamaEmbeddings(model="gemma:2b")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)
vectordb

C:\Users\HP\AppData\Local\Temp\ipykernel_19416\91020916.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model="gemma:2b")


In [6]:
## QUERY THE DB
query = "What is the speech about?"
docs = vectordb.similarity_search(query=query)
docs

[Document(id='9fd1744c-d9e9-4d38-9e9a-0baca4f1623c', metadata={'source': 'speech.txt'}, page_content='here and there and without countenance except from a lawless and malignant few.'),
 Document(id='3b3525a1-a801-45dd-8620-d2a5a55f8ab3', metadata={'source': 'speech.txt'}, page_content='and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='0aef7fe2-9673-42cd-8e22-a337f534a343', metadata={'source': 'speech.txt'}, page_content='we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='465a7459-f93c-455f-9ffe

In [7]:
## saving to disk
vectordb = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory="./chroma_db")

In [8]:
## load from disk
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs = db2.similarity_search(query=query)
docs

[Document(id='f8e3e8d3-22d4-4cd6-a816-84b78f6a6b83', metadata={'source': 'speech.txt'}, page_content='here and there and without countenance except from a lawless and malignant few.'),
 Document(id='8f395d56-60d3-4476-bb83-cad5855d30b0', metadata={'source': 'speech.txt'}, page_content='and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='3ff421fa-3d59-406c-934f-50ea668b0577', metadata={'source': 'speech.txt'}, page_content='we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='dcc570ea-a39e-401d-bb27

In [9]:
## similarity search with score
docs = vectordb.similarity_search_with_score(query=query)
docs

[(Document(id='f8e3e8d3-22d4-4cd6-a816-84b78f6a6b83', metadata={'source': 'speech.txt'}, page_content='here and there and without countenance except from a lawless and malignant few.'),
  2530.716796875),
 (Document(id='8f395d56-60d3-4476-bb83-cad5855d30b0', metadata={'source': 'speech.txt'}, page_content='and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  2559.80029296875),
 (Document(id='3ff421fa-3d59-406c-934f-50ea668b0577', metadata={'source': 'speech.txt'}, page_content='we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last fre

In [10]:
## Retriever option
retriever = vectordb.as_retriever()
retriever_result = retriever.invoke(query)
retriever_result[0]

Document(id='f8e3e8d3-22d4-4cd6-a816-84b78f6a6b83', metadata={'source': 'speech.txt'}, page_content='here and there and without countenance except from a lawless and malignant few.')